In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


####Data Reading

In [0]:
df=spark.read.format("parquet")\
    .load("abfss://bronze@datalakedatabricksram.dfs.core.windows.net/products")


In [0]:
df.display()

In [0]:
df=df.drop("_rescued_data")

In [0]:
df.createOrReplaceTempView("products")

1

In [0]:
%sql
create or replace function databricks_catalog.bronze.discount_fun(p_price Double) 
returns double
language sql
return p_price*0.90

In [0]:
%sql
select product_id, price, databricks_catalog.bronze.discount_fun(price) as discounted_price from products

In [0]:
%sql
use catalog databricks_catalog

In [0]:
df=df.withColumn("discounted_price",expr("databricks_catalog.bronze.discount_fun(price)"))


In [0]:
df.display()

In [0]:
df.write.format('delta')\
    .mode("overwrite")\
        .option("path","abfss://silver@datalakedatabricksram.dfs.core.windows.net/products")\
            .save()


In [0]:
%sql 
create table if not exists databricks_catalog.silver.products_silver
using delta
location 'abfss://silver@datalakedatabricksram.dfs.core.windows.net/products'